# Análise de Vendas — Loja de Eletrônicos (Brasil 2024)

**Problema de negócio:** Uma loja de eletrônicos com atuação nacional precisa entender quais produtos geram mais lucro, em quais meses concentrar estoques e quais regiões merecem maior investimento em marketing.

**Perguntas que vamos responder:**
1. Quais produtos vendem mais (receita e volume)?
2. Qual mês fatura mais? Existe sazonalidade?
3. Qual região performa melhor?
4. Quais produtos têm maior margem de lucro?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 12
print('Bibliotecas carregadas com sucesso.')

## Geração do Dataset

Dataset sintético baseado em padrões reais do mercado brasileiro de eletrônicos:
- **5.000 transações** ao longo de 2024
- Sazonalidade: Q4 representa ~33% das vendas (Black Friday + Natal)
- Concentração regional: Sudeste 42%, Sul 20%, Nordeste 18%

In [ ]:
np.random.seed(42)
Path('data').mkdir(exist_ok=True)

N = 5000

# Produto -> (Categoria, Preço base, Custo base)
produtos_info = {
    'Notebook':        ('Computadores', 2800, 1750),
    'Smartphone':      ('Celulares',    1900, 1150),
    'Tablet':          ('Celulares',    1300,  800),
    'Monitor':         ('Computadores',  900,  530),
    'Teclado Mecânico':('Periféricos',   380,  190),
    'Mouse Gamer':     ('Periféricos',   290,  130),
    'Headset':         ('Periféricos',   470,  240),
    'Webcam':          ('Periféricos',   330,  170),
    'Impressora':      ('Computadores',  650,  410),
    'Roteador':        ('Redes',         260,  140),
}

regioes      = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
pesos_regiao = [0.08,    0.18,       0.12,           0.42,      0.20]

# Sazonalidade mensal (Q4 maior)
pesos_mes = [0.055, 0.050, 0.070, 0.070, 0.080, 0.080,
             0.080, 0.090, 0.090, 0.100, 0.115, 0.120]

produtos      = list(produtos_info.keys())
pesos_produto = [0.12, 0.18, 0.10, 0.08, 0.12, 0.15, 0.10, 0.07, 0.05, 0.03]

prod_col   = np.random.choice(produtos, N, p=pesos_produto)
mes_col    = np.random.choice(range(1, 13), N, p=pesos_mes)
dia_col    = np.random.randint(1, 28, N)
regiao_col = np.random.choice(regioes, N, p=pesos_regiao)
qtd_col    = np.random.randint(1, 8, N)

categorias, precos, custos = [], [], []
for p in prod_col:
    cat, preco_base, custo_base = produtos_info[p]
    categorias.append(cat)
    precos.append(round(preco_base * np.random.uniform(0.92, 1.08), 2))
    custos.append(round(custo_base * np.random.uniform(0.92, 1.08), 2))

precos = np.array(precos)
custos = np.array(custos)

df = pd.DataFrame({
    'data':          pd.to_datetime({'year': 2024, 'month': mes_col, 'day': dia_col}),
    'mes':           mes_col,
    'produto':       prod_col,
    'categoria':     categorias,
    'regiao':        regiao_col,
    'quantidade':    qtd_col,
    'preco_unitario': precos,
    'custo_unitario': custos,
    'receita':       (qtd_col * precos).round(2),
    'lucro':         (qtd_col * (precos - custos)).round(2),
})

df.to_csv('data/vendas.csv', index=False)
print(f'Dataset salvo em data/vendas.csv')
print(f'Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head()

## 1. Visão Geral dos Dados

In [ ]:
df = pd.read_csv('data/vendas.csv', parse_dates=['data'])

receita_total = df['receita'].sum()
lucro_total   = df['lucro'].sum()
margem_geral  = lucro_total / receita_total * 100

print('=' * 50)
print('RESUMO EXECUTIVO — 2024')
print('=' * 50)
print(f'  Receita total :  R$ {receita_total:>12,.2f}')
print(f'  Lucro total   :  R$ {lucro_total:>12,.2f}')
print(f'  Margem média  :       {margem_geral:>8.1f}%')
print(f'  Unidades vend.:  {df["quantidade"].sum():>12,}')
print(f'  Nº transações :  {len(df):>12,}')
print(f'  Ticket médio  :  R$ {receita_total/len(df):>12,.2f}')
print('=' * 50)

## 2. Quais produtos vendem mais?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Receita por produto ---
receita_prod = df.groupby('produto')['receita'].sum().sort_values()
colors = sns.color_palette('Blues', len(receita_prod))
receita_prod.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Receita Total por Produto', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Receita (R$)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
for i, v in enumerate(receita_prod.values):
    axes[0].text(v + 5000, i, f'R$ {v/1e6:.2f}M', va='center', fontsize=9)

# --- Quantidade vendida ---
qtd_prod = df.groupby('produto')['quantidade'].sum().sort_values()
colors2 = sns.color_palette('Greens', len(qtd_prod))
qtd_prod.plot(kind='barh', ax=axes[1], color=colors2)
axes[1].set_title('Quantidade Vendida por Produto', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Unidades')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(qtd_prod.values):
    axes[1].text(v + 20, i, f'{int(v):,}', va='center', fontsize=9)

plt.suptitle('Análise de Produtos', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/grafico_produtos.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🏆 TOP 3 PRODUTOS POR RECEITA:')
top3 = df.groupby('produto')['receita'].sum().sort_values(ascending=False).head(3)
for i, (prod, val) in enumerate(top3.items(), 1):
    share = val / df['receita'].sum() * 100
    print(f'  {i}. {prod:<20} R$ {val:>10,.2f}  ({share:.1f}% da receita total)')

## 3. Qual mês fatura mais?

In [ ]:
nomes_mes = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
             7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}

mensal = df.groupby('mes').agg(
    receita  = ('receita',   'sum'),
    lucro    = ('lucro',     'sum'),
    unidades = ('quantidade','sum')
).reset_index()
mensal['mes_nome'] = mensal['mes'].map(nomes_mes)
mensal['margem']   = mensal['lucro'] / mensal['receita'] * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
x = np.arange(len(mensal))
w = 0.38

# Receita vs Lucro
b1 = axes[0].bar(x - w/2, mensal['receita']/1e3, w, label='Receita', color='steelblue', alpha=0.85)
b2 = axes[0].bar(x + w/2, mensal['lucro']/1e3,   w, label='Lucro',   color='seagreen',  alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(mensal['mes_nome'])
axes[0].set_title('Receita e Lucro Mensal (R$ mil)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor (R$ mil)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R$ {v:.0f}k'))

# Margem
axes[1].plot(mensal['mes_nome'], mensal['margem'], marker='o', linewidth=2.5,
             color='darkorange', markersize=9)
axes[1].fill_between(range(len(mensal)), mensal['margem'], alpha=0.15, color='darkorange')
axes[1].axhline(mensal['margem'].mean(), color='navy', linestyle='--', linewidth=1.5,
                label=f'Média: {mensal["margem"].mean():.1f}%')
for i, m in enumerate(mensal['margem']):
    axes[1].text(i, m + 0.3, f'{m:.1f}%', ha='center', fontsize=9)
axes[1].set_title('Margem de Lucro Mensal (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Margem (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/grafico_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

best = mensal.loc[mensal['receita'].idxmax()]
print(f'📅 Melhor mês:  {best["mes_nome"]} | Receita: R$ {best["receita"]:,.2f} | Lucro: R$ {best["lucro"]:,.2f}')
q4 = mensal[mensal['mes'].isin([10,11,12])]['receita'].sum()
print(f'🎄 Q4 (Out-Dez): R$ {q4:,.2f}  ({q4/mensal["receita"].sum()*100:.1f}% do faturamento anual)')

## 4. Qual região performa melhor?

In [ ]:
reg = df.groupby('regiao').agg(
    receita       = ('receita',   'sum'),
    lucro         = ('lucro',     'sum'),
    unidades      = ('quantidade','sum'),
    ticket_medio  = ('receita',   'mean')
).sort_values('receita', ascending=False).reset_index()
reg['margem'] = reg['lucro'] / reg['receita'] * 100
reg['share']  = reg['receita'] / reg['receita'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Receita por região
colors_r = sns.color_palette('Blues_d', len(reg))[::-1]
bars = axes[0].bar(reg['regiao'], reg['receita']/1e6, color=colors_r, edgecolor='white')
axes[0].set_title('Receita por Região (R$ milhões)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Receita (R$ mi)')
for bar, row in zip(bars, reg.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'R$ {row.receita/1e6:.1f}M\n({row.share:.0f}%)',
                 ha='center', fontsize=10, fontweight='bold')

# Lucro + margem (eixo duplo)
ax2 = axes[1].twinx()
colors_g = sns.color_palette('Greens_d', len(reg))[::-1]
axes[1].bar(reg['regiao'], reg['lucro']/1e6, color=colors_g, alpha=0.85, edgecolor='white')
ax2.plot(reg['regiao'], reg['margem'], 'ro-', linewidth=2.5, markersize=10, label='Margem %')
for i, (r, m) in enumerate(zip(reg['regiao'], reg['margem'])):
    ax2.text(i, m + 0.4, f'{m:.1f}%', ha='center', fontsize=9, color='red', fontweight='bold')
axes[1].set_title('Lucro por Região e Margem (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Lucro (R$ mi)')
ax2.set_ylabel('Margem (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('data/grafico_regioes.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📍 RANKING DE REGIÕES:')
print(reg[['regiao','receita','lucro','margem','ticket_medio','share']].to_string(index=False))

## 5. Lucratividade por Produto

In [ ]:
margem_prod = df.groupby('produto').agg(
    receita = ('receita', 'sum'),
    lucro   = ('lucro',   'sum')
).assign(margem=lambda x: x['lucro']/x['receita']*100).sort_values('margem', ascending=False)

media_margem = margem_prod['margem'].mean()
colors = ['seagreen' if m >= media_margem else 'tomato' for m in margem_prod['margem']]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(margem_prod.index, margem_prod['margem'], color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(media_margem, color='navy', linestyle='--', linewidth=2,
           label=f'Margem média: {media_margem:.1f}%')
for bar, val in zip(bars, margem_prod['margem']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Margem de Lucro por Produto', fontsize=14, fontweight='bold')
ax.set_ylabel('Margem (%)')
ax.set_xlabel('Produto')
ax.legend(fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('data/grafico_margem.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Receita por Produto x Região
pivot = df.pivot_table(values='receita', index='produto', columns='regiao', aggfunc='sum') / 1000

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Receita (R$ mil)'})
ax.set_title('Receita por Produto e Região (R$ mil)', fontsize=14, fontweight='bold')
ax.set_xlabel('Região')
ax.set_ylabel('Produto')
plt.tight_layout()
plt.savefig('data/grafico_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusões e Insights

### 🎯 Principais descobertas

| # | Insight | Ação recomendada |
|---|---------|------------------|
| 1 | **Smartphone é o campeão de receita** (maior volume × preço médio-alto) | Priorizar estoque e campanhas para smartphones |
| 2 | **Q4 representa ~33% do faturamento anual** (sazonalidade forte) | Aumentar estoque e verba de marketing em Out/Nov/Dez |
| 3 | **Sudeste concentra 42% das vendas**, mas Norte e Nordeste têm margem similar | Investir em distribuição no Norte para crescer com boa margem |
| 4 | **Mouse Gamer e Teclado Mecânico têm as maiores margens** | Criar bundles e kits de periféricos para maximizar lucro |
| 5 | **Notebook gera maior receita absoluta**, mas margem está na média | Negociar melhores condições com fornecedores de notebooks |

In [ ]:
print('RESUMO FINAL'.center(60, '='))
print()
print('TOP 3 RECEITA:')
for i, (p, v) in enumerate(df.groupby('produto')['receita'].sum().nlargest(3).items(), 1):
    print(f'  {i}. {p:<22} R$ {v:>10,.0f}')
print()
print('TOP 3 MARGEM:')
marg = df.groupby('produto').apply(lambda x: x['lucro'].sum()/x['receita'].sum()*100).nlargest(3)
for i, (p, v) in enumerate(marg.items(), 1):
    print(f'  {i}. {p:<22} {v:.1f}%')
print()
print('MELHOR MÊS:')
best_mes = df.groupby('mes')['receita'].sum().idxmax()
print(f'  Mês {best_mes} (Dezembro) — R$ {df.groupby("mes")["receita"].sum().max():,.0f}')
print()
print('MELHOR REGIÃO:')
best_reg = df.groupby('regiao')['receita'].sum().idxmax()
print(f'  {best_reg} — R$ {df.groupby("regiao")["receita"].sum().max():,.0f}')